# 02 — Silver Layer: Cleaning & Validation

**Purpose:** Read raw Bronze data, apply business rules, and produce a clean, trusted Silver Delta table.

**Medallion layer:** 🥈 Silver — cleaned, validated, enriched — the canonical layer

**What you will learn in this notebook:**
- How to read a partitioned Delta table
- How to apply and document business cleaning rules
- How to deduplicate on a business key
- How to add derived/enrichment columns
- How to use window functions for ranking
- How to track removal counts for auditing

---
**Input:**  Bronze Delta table (`data/bronze/`)  
**Output:** Silver Delta table (`data/silver/`)  

## 0. Setup

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType
from pyspark.sql.window import Window

BRONZE_PATH = os.path.join(PROJECT_ROOT, 'data', 'bronze')
SILVER_PATH = os.path.join(PROJECT_ROOT, 'data', 'silver')

YEAR  = 2024
MONTH = 1

os.makedirs(SILVER_PATH, exist_ok=True)
print(f"Bronze : {BRONZE_PATH}")
print(f"Silver : {SILVER_PATH}")

In [ ]:
spark = (
    SparkSession.builder
    .appName("nyc-taxi-silver")
    .master("local[*]")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready")

## 1. Read from Bronze

We filter to a single partition (year + month) for efficiency.  
In production this would process the latest partition only.

In [ ]:
df = (
    spark.read.format("delta").load(BRONZE_PATH)
    .filter(
        (F.col("pipeline_year")  == YEAR) &
        (F.col("pipeline_month") == MONTH)
    )
)

bronze_count = df.count()
print(f"Rows read from Bronze: {bronze_count:,}")
print(f"Columns              : {len(df.columns)}")

## 2. Rename Columns and Cast Types

Raw TLC data mixes CamelCase and snake_case. We standardise everything  
to snake_case and cast to the correct types at the same time.

**Interview tip:** Always do renames and casts in the same step — it keeps  
the lineage clear and avoids redundant passes over the data.

In [ ]:
df = (
    df
    # Rename to snake_case
    .withColumnRenamed("VendorID",     "vendor_id")
    .withColumnRenamed("RatecodeID",   "rate_code_id")
    .withColumnRenamed("PULocationID", "pickup_location_id")
    .withColumnRenamed("DOLocationID", "dropoff_location_id")
    # Cast types
    .withColumn("vendor_id",        F.col("vendor_id").cast(IntegerType()))
    .withColumn("passenger_count",  F.col("passenger_count").cast(IntegerType()))
    .withColumn("trip_distance",    F.col("trip_distance").cast(DoubleType()))
    .withColumn("fare_amount",      F.col("fare_amount").cast(DoubleType()))
    .withColumn("tip_amount",       F.col("tip_amount").cast(DoubleType()))
    .withColumn("total_amount",     F.col("total_amount").cast(DoubleType()))
    .withColumn("payment_type",     F.col("payment_type").cast(IntegerType()))
    .withColumn("tpep_pickup_datetime",
                F.col("tpep_pickup_datetime").cast(TimestampType()))
    .withColumn("tpep_dropoff_datetime",
                F.col("tpep_dropoff_datetime").cast(TimestampType()))
)

print("Types after casting:")
df.select("vendor_id", "fare_amount", "trip_distance",
          "tpep_pickup_datetime", "payment_type").printSchema()

## 3. Filter Invalid Rows

We apply 5 business rules. Each removal count is tracked separately  
so we can see exactly what was cleaned and why — essential for auditing.

In [ ]:
# ── Rule 1: fare_amount must be positive ──────────────────────────
# Negative fares indicate refunds or data entry errors
before_r1 = df.count()
df = df.filter(F.col("fare_amount") > 0)
removed_r1 = before_r1 - df.count()
print(f"Rule 1 — negative fares removed  : {removed_r1:,}")

In [ ]:
# ── Rule 2: trip_distance must be non-negative ────────────────────
before_r2 = df.count()
df = df.filter(F.col("trip_distance") >= 0)
removed_r2 = before_r2 - df.count()
print(f"Rule 2 — negative distances removed: {removed_r2:,}")

In [ ]:
# ── Rule 3: pickup datetime must be valid ─────────────────────────
before_r3 = df.count()
df = df.filter(
    F.col("tpep_pickup_datetime").isNotNull() &
    (F.col("tpep_pickup_datetime") <= F.current_timestamp())
)
removed_r3 = before_r3 - df.count()
print(f"Rule 3 — invalid pickup datetimes  : {removed_r3:,}")

In [ ]:
# ── Rule 4: dropoff must be after pickup ──────────────────────────
before_r4 = df.count()
df = df.filter(
    F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime")
)
removed_r4 = before_r4 - df.count()
print(f"Rule 4 — dropoff before pickup     : {removed_r4:,}")

In [ ]:
# ── Rule 5: passenger count must be 1–8 ──────────────────────────
before_r5 = df.count()
df = df.filter(F.col("passenger_count").between(1, 8))
removed_r5 = before_r5 - df.count()
print(f"Rule 5 — invalid passenger count   : {removed_r5:,}")

In [ ]:
# ── Cleaning summary ──────────────────────────────────────────────
silver_after_filter = df.count()
total_removed = bronze_count - silver_after_filter

print(f"\nCleaning Summary")
print("-" * 45)
print(f"  Bronze input         : {bronze_count:>10,}")
print(f"  Negative fares       : {removed_r1:>10,}")
print(f"  Negative distances   : {removed_r2:>10,}")
print(f"  Invalid pickup time  : {removed_r3:>10,}")
print(f"  Dropoff before pickup: {removed_r4:>10,}")
print(f"  Invalid passengers   : {removed_r5:>10,}")
print("-" * 45)
print(f"  Total removed        : {total_removed:>10,}  ({total_removed/bronze_count*100:.2f}%)")
print(f"  After filtering      : {silver_after_filter:>10,}")

## 4. Deduplicate

The **business key** for a taxi trip is `vendor_id + tpep_pickup_datetime`.  
This combination uniquely identifies one trip from one vendor.

**Interview tip:** Always define your business key explicitly and document why.  
Using `dropDuplicates()` without arguments removes exact row duplicates only —  
business-key deduplication is different and more important.

In [ ]:
before_dedup = df.count()
df = df.dropDuplicates(["vendor_id", "tpep_pickup_datetime"])
dupes_removed = before_dedup - df.count()

print(f"Rows before dedup : {before_dedup:,}")
print(f"Duplicates removed: {dupes_removed:,}")
print(f"Rows after dedup  : {df.count():,}")

## 5. Add Derived Columns

Derived columns are computed from existing columns and added to Silver.  
They make Gold aggregations simpler and avoid repeated computation.

**What we add:**
- `trip_duration_minutes` — from pickup/dropoff timestamps
- `fare_per_mile` — useful for pricing analysis  
- `pickup_date`, `pickup_hour`, `pickup_dow` — for time-series aggregations
- `payment_type_label` — human-readable payment type
- `is_outlier` — flag suspiciously high fares (kept, not removed)

In [ ]:
# Payment type mapping (from TLC data dictionary)
PAYMENT_MAP = {
    1: "Credit card", 2: "Cash", 3: "No charge",
    4: "Dispute",     5: "Unknown", 6: "Voided trip"
}
payment_mapping = F.create_map(
    *[val for pair in
      [(F.lit(k), F.lit(v)) for k, v in PAYMENT_MAP.items()]
      for val in pair]
)

df = (
    df
    # Trip duration in minutes
    .withColumn(
        "trip_duration_minutes",
        F.round(
            (F.unix_timestamp("tpep_dropoff_datetime") -
             F.unix_timestamp("tpep_pickup_datetime")) / 60, 2
        )
    )
    # Fare per mile (null when distance is 0 — avoid division by zero)
    .withColumn(
        "fare_per_mile",
        F.when(
            F.col("trip_distance") > 0,
            F.round(F.col("fare_amount") / F.col("trip_distance"), 2)
        ).otherwise(None)
    )
    # Date parts for aggregation
    .withColumn("pickup_date",  F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour",  F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_dow",   F.dayofweek("tpep_pickup_datetime"))  # 1=Sun, 7=Sat
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_year",  F.year("tpep_pickup_datetime"))
    # Payment label
    .withColumn("payment_type_label", payment_mapping[F.col("payment_type")])
    # Outlier flag — fares above $500 kept but flagged
    .withColumn("is_outlier", F.col("fare_amount") > 500)
    # Silver audit
    .withColumn("silver_processed_at", F.current_timestamp())
)

print("Derived columns added successfully")
df.select(
    "fare_amount", "trip_distance", "fare_per_mile",
    "trip_duration_minutes", "pickup_hour", "payment_type_label"
).show(5)

## 6. Explore the Cleaned Data

Always explore Silver before writing — confirm the cleaning worked as expected.

In [ ]:
# Trip duration distribution
print("Trip duration statistics (minutes):")
df.select("trip_duration_minutes").describe().show()

In [ ]:
# Fare per mile — useful pricing sanity check
print("Fare per mile (excluding outliers):")
df.filter(F.col("is_outlier") == False) \
  .select("fare_per_mile").describe().show()

In [ ]:
# Trips by hour of day — demand pattern preview
print("Trip count by hour of day:")
df.groupBy("pickup_hour") \
  .count() \
  .orderBy("pickup_hour") \
  .show(24, truncate=False)

In [ ]:
# Payment type breakdown
print("Payment type distribution:")
df.groupBy("payment_type_label") \
  .agg(
      F.count("*").alias("trips"),
      F.round(F.avg("tip_amount"), 2).alias("avg_tip")
  ) \
  .orderBy("trips", ascending=False) \
  .show()

In [ ]:
# Outlier count
outliers = df.filter(F.col("is_outlier") == True).count()
print(f"Outlier trips (fare > $500): {outliers:,}  ({outliers/df.count()*100:.4f}%)")
print("  These are KEPT in Silver but EXCLUDED from Gold aggregations")

## 7. Write to Silver Delta Table

In [ ]:
silver_count = df.count()
print(f"Writing {silver_count:,} rows to Silver...")

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("pickup_year", "pickup_month")
    .save(SILVER_PATH)
)

print(f"✓ Silver write complete → {SILVER_PATH}")

## 8. Silver Data Quality Checks

In [ ]:
df_s = spark.read.format("delta").load(SILVER_PATH)
total = df_s.count()

checks = [
    ("No negative fares",
     df_s.filter(F.col("fare_amount") <= 0).count() == 0),
    ("No null pickup datetimes",
     df_s.filter(F.col("tpep_pickup_datetime").isNull()).count() == 0),
    ("No invalid passenger counts",
     df_s.filter(~F.col("passenger_count").between(1, 8)).count() == 0),
    ("trip_duration_minutes column exists",
     "trip_duration_minutes" in df_s.columns),
    ("fare_per_mile column exists",
     "fare_per_mile" in df_s.columns),
    ("pickup_year column exists",
     "pickup_year" in df_s.columns),
]

print("Silver Data Quality Report")
print("-" * 45)
for name, passed in checks:
    print(f"  {'✓' if passed else '✗'}  {name}")
print("-" * 45)
print(f"  Rows in Silver: {total:,}")
print(f"  Removal rate  : {(bronze_count - total) / bronze_count * 100:.2f}%")
print(f"  {'ALL CHECKS PASSED ✓' if all(p for _, p in checks) else 'SOME CHECKS FAILED ✗'}")

## ✅ Summary

| Step | Action | Rows |
|------|--------|------|
| Read Bronze | Load raw data | ~3,000,000 |
| Rename & cast | Standardise columns and types | — |
| Filter rules | Remove 5 categories of invalid rows | −~50,000 |
| Deduplicate | Remove business-key duplicates | −~500 |
| Derived columns | Add 8 enrichment columns | — |
| Write Silver | Partitioned Delta table | ~2,950,000 |

**Next:** Open `03_gold_exploration.ipynb` to build analytics aggregations.